In [1]:
from pathlib import Path
from typing import Tuple

import cv2 
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from tqdm import tqdm
import math

### Watch the videos

In [2]:
input_dir = Path('input')

pedestrians_path = str(input_dir / 'pedestrians.avi')
pedestrians_box_path = str(input_dir / 'pedestrians.txt')

debate_path = str(input_dir / 'pres_debate.avi')
debate_box_path = str(input_dir / 'pres_debate.txt')

noisy_debate_path = str(input_dir / 'noisy_debate.avi')
noisy_debate_box_path = str(input_dir / 'noisy_debate.txt')

debate_hand_box_path = str(input_dir / 'pres_debate_hand.txt')

In [3]:
def get_video_frames(video_path, gray=True):
    video = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = video.read()
        if not ret:
            break
        if gray:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        # else:
            # frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    video.release()
    return frames


def get_center_and_sizes(video_path):
    box_path = video_path[:-3] + "txt"  
    with open(box_path, 'r') as f:
        lines = f.readlines()
        data = []
        for line in lines:
            numbers = list(map(lambda x: int(float(x)), line.split()))
            data.append(numbers)


    center_and_sizes = {"x": data[0][0], "y": data[0][1], "w": data[1][0], "h": data[1][1]}
    x,y,w,h = center_and_sizes.values()
    x,y = x+w//2, y+w//2
    return x,y,w,h

def show_video(video_path):
    video = cv2.VideoCapture(video_path)
    while True:
        ret, frame = video.read()
        if not ret:
            break
        cv2.imshow('Frame', frame)
        # Press Q on keyboard to exit the display window
        if cv2.waitKey(25) & 0xFF == ord('q'):
            break
    video.release()
    cv2.destroyAllWindows()

In [4]:
frames_debate = get_video_frames(debate_path)
frames_debate_bgr = get_video_frames(debate_path, gray=False)

In [91]:
def get_patch(image, x, y, w, h):
    return image[y - h//2:y + h//2 + 1, x - w//2: x + w//2 + 1]

def is_image_gray_scale(image):
    return len(image.shape) == 2 or (len(image.shape) == 3 and image.shape[2] == 1) 

class TrackBox:
    def __init__(self, image, x,y,w,h) -> None:
        self.image = image
        self.x = x
        self.y = y
        self.w = w
        self.h = h
        self.patch = self.image[y - h//2:y + h//2 + 1, x - w//2: x + w//2 + 1]
    
    def get_components(self):
        return self.x, self.y, self.w, self.h
    def get_patch(self):
        return self.patch
    def get_top_left(self):
        return (self.x-self.w//2, self.y-self.h//2)
    def get_bottom_right(self):
        return (self.x+self.w//2 + 1, self.y + self.h//2 +1)
    
class ParticleFilterState():
    def __init__(self, frame, particles, weights, previous_track_box) -> None:
        self.frame = frame
        self.particles = particles
        self.weights = weights
        self.previous_track_box = previous_track_box

    def get_top_track_boxes(self):
        pass
    
    def get_best_track_box(self):
        x, y = self.particles[np.argmax(self.weights)]
        w, h = self.previous_track_box.w, self.previous_track_box.h
        return TrackBox(self.frame, x, y, w, h)
    
    def get_best_patch(self):
        x, y = self.particles[np.argmax(self.weights)]
        w, h = self.previous_track_box.w, self.previous_track_box.h
        return get_patch(self.frame, x, y, w, h)
    
    def __get_frame_with_box(self, frame, color):
        track_box = self.get_best_track_box()
        frame_colored = frame.copy()
        if is_image_gray_scale(frame_colored):
            frame_colored = cv2.cvtColor(frame_colored, cv2.COLOR_GRAY2RGB)
        frame_with_box = cv2.rectangle(frame_colored, track_box.get_top_left(), track_box.get_bottom_right(), color=color, thickness=3)
        frame_with_box_and_center_point = cv2.circle(frame_with_box, (track_box.x, track_box.y), radius=3, color=color, thickness=10)
        return frame_with_box_and_center_point
    
    def get_frame_with_box(self, color=[255,0,0]):
        return self.__get_frame_with_box(self.frame, color)        
    
    def __get_frame_with_particles(self, frame):
        frame_colored = frame.copy()
        if is_image_gray_scale(frame_colored):
            frame_colored = cv2.cvtColor(frame_colored, cv2.COLOR_GRAY2RGB)

        # Convert normalized weights to a color map (0 to 255 range)
        indices = np.argsort(self.weights)
        self.weights = self.weights[indices]
        self.particles = self.particles[indices]
        colors = (self.weights * 255).astype(np.uint8)
        colors = cv2.applyColorMap(colors, cv2.COLORMAP_JET)

        for i,((x, y), color) in enumerate(zip(self.particles, colors)):
            frame_colored = cv2.circle(frame_colored, (int(x), int(y)), radius=(int(10*self.weights[i])), color=list(int(c) for c in color[0]), thickness=2)

        return frame_colored
    
    def get_frame_with_particles(self):
        return self.__get_frame_with_particles(self.frame)
    
    def get_frame_with_particles_and_box(self, color=[255,0,0]):
        frame_with_box = self.get_frame_with_box(color)
        return self.__get_frame_with_particles(frame_with_box)
    

class ParticleFilterStateIIR(ParticleFilterState):
    def __init__(self, frame, particles, weights, previous_track_box, clipping_fn, alpha=0.8) -> None:
        super().__init__(frame, particles, weights, previous_track_box)
        self.alpha = alpha
        self.clipping_fn = clipping_fn

    def __get_new_particle(self, particle):
        x,y = particle
        alpha = self.alpha
        x_prev,y_prev = self.previous_track_box.x, self.previous_track_box.y
        x,y = (alpha*x + (1-alpha)*x_prev, alpha*y + (1-alpha)*y_prev)
        x,y = self.clipping_fn(self.frame, self.previous_track_box, np.array([(x,y)]))[0]
        return int(x),int(y)
    
    def get_best_track_box(self):
        x, y = self.particles[np.argmax(self.weights)]
        w, h = self.previous_track_box.w, self.previous_track_box.h
        x,y = self.__get_new_particle((x,y))
        return TrackBox(self.frame, x, y, w, h)
    
    def get_best_patch(self):
        return self.get_best_track_box().get_patch()
    


class ParticleFilterAlgo():
    def __init__(self, frames, initial_particles_fn, sampling_fn ,diffuse_fn, measure_fn, clipping_fn, num_particles=200, num_frames=150) -> None:
        self.frames = frames
        self.initial_particles_fn = initial_particles_fn
        self.sampling_fn = sampling_fn
        self.diffuse_fn = diffuse_fn
        self.measure_fn = measure_fn
        self.clipping_fn = clipping_fn
        self.num_particles = num_particles
        self.num_frames = num_frames

    def _get_track_box_from_path(self, image, track_box_path):
        box_path = track_box_path  
        with open(box_path, 'r') as f:
            lines = f.readlines()
            data = []
            for line in lines:
                numbers = list(map(lambda x: int(float(x)), line.split()))
                data.append(numbers)


        top_left_and_sizees = {"x": data[0][0], "y": data[0][1], "w": data[1][0], "h": data[1][1]}
        x,y,w,h = top_left_and_sizees.values()
        x,y = x+w//2, y+h//2
        
        return TrackBox(image, x, y, w, h)
        

    def _create_initial_state(self, initial_track_box_path):

        initial_track_box = self._get_track_box_from_path(self.frames[0], initial_track_box_path)
        particles = self.initial_particles_fn(self.num_particles, initial_track_box)
        particles = self.clipping_fn(self.frames[0], initial_track_box, particles)
        weights = self.measure_fn(self.frames[0], initial_track_box, particles)
        return ParticleFilterState(self.frames[0], particles, weights, initial_track_box)

    def apply_particle_filtering(self, initial_track_box_path):
        curr_state = self._create_initial_state(initial_track_box_path)
        particles_states = [curr_state]
        
        for new_frame in tqdm(self.frames[1: self.num_frames], total=self.num_frames, initial=1):
            new_particles_state = self.advance_round(new_frame, curr_state)
            particles_states.append(new_particles_state)
            curr_state = new_particles_state
        return particles_states

    def advance_round(self, new_frame, curr_state: ParticleFilterState):
        prev_track_box = curr_state.get_best_track_box()
        new_frame = new_frame.copy()
        
        samples = self.sampling_fn(curr_state.particles, curr_state.weights)
        samples = self.diffuse_fn(samples)
        samples = self.clipping_fn(new_frame, prev_track_box, samples)
        weights = self.measure_fn(new_frame, prev_track_box, samples)
        return ParticleFilterState(new_frame, samples, weights, prev_track_box)
    

    def __convert_particle_state_to_IIR_state(self, state:ParticleFilterState, alpha)->ParticleFilterStateIIR:
        return ParticleFilterStateIIR(state.frame, state.particles, state.weights, state.previous_track_box, self.clipping_fn, alpha)
    
    def apply_particle_filter_IIR(self, initial_track_box_path, alpha=0.5):
        curr_state = self._create_initial_state(initial_track_box_path)
        curr_state = self.__convert_particle_state_to_IIR_state(curr_state, alpha)
        particles_states = [curr_state]
        
        for new_frame in tqdm(self.frames[1: self.num_frames], total=self.num_frames, initial=1):
            new_particles_state = self.advance_round(new_frame, curr_state)
            new_particles_state = self.__convert_particle_state_to_IIR_state(new_particles_state, alpha)
            particles_states.append(new_particles_state)
            curr_state = new_particles_state

        return particles_states



In [92]:
# Expect particles to be (x,y) position (col,row)
def compute_gaussian_mse_map_particle_set(image, prev_track_box: TrackBox, particles, sigma_mse:float=10):
    mse_map = np.full_like(image, np.inf, dtype=np.float32)
    
    for (x,y) in particles:
        patch_img = get_patch(image, x, y, prev_track_box.w, prev_track_box.h)
        diff = (np.astype(patch_img, np.float32) - np.astype(prev_track_box.get_patch(), np.float32))
        mse = np.mean(diff ** 2)
        mse_map[y,x] = mse
        # print(x,y," is:", mse)
        # print(x,y," is:", np.power(math.e, -1 * (mse/(2*(sigma_mse**2)))))

    
    gaus_mse_map = np.power(math.e, -(mse_map/(2*(sigma_mse**2))))
    return gaus_mse_map


def diffuse(particles, noise_sigma=3):
    x_noises = np.random.normal(loc=0, scale=noise_sigma, size=len(particles))
    y_noises = np.random.normal(loc=0, scale=noise_sigma, size=len(particles))
    diffused = np.array([(pt[0] + x_noises[i] , pt[1] + y_noises[i]) for i,pt in enumerate(particles)])
    return np.round(diffused).astype(int)

def create_particles(num_particles, track_box: TrackBox):
    x,y,w,h = track_box.get_components()
    # Assuming 3-sigma rule (99.7% of values fall within width)
    sigma_x = w / 6  
    sigma_y = h / 6

    # Sample particles from a normal distribution
    xs = np.random.normal(loc=x, scale=sigma_x, size=num_particles)
    ys = np.random.normal(loc=y, scale=sigma_y, size=num_particles)

    xs = np.round(xs).astype(int)
    ys = np.round(ys).astype(int)    

    particles = np.column_stack((xs, ys))

    return particles

def clip_into_frame(image, track_box: TrackBox, particles):
    xs = particles[:,0]
    ys = particles[:,1]
    x_min, x_max = 0 + track_box.w//2, image.shape[1] - track_box.w//2 - 1
    y_min, y_max = 0 + track_box.h//2, image.shape[0] - track_box.h//2 - 1

    xs = np.clip(xs, x_min, x_max)
    ys = np.clip(ys, y_min, y_max)

    return np.column_stack((xs, ys))

def sample_particles(particles, weights):
    # samples_distribution = np.random.multinomial(num_samples, weights) # number of times sampled each sample
    # samples = [particles[i] for i in range(len(samples_distribution)) for _ in range(samples_distribution[i])]
    # print(samples_distribution)
    indices = np.random.choice(np.arange(len(particles)), size=len(particles), p=weights, replace=True)
    samples = particles[indices]
    return samples

def get_measure_fn(sigma_mse=10):
    def get_weights(image, prev_track_box: TrackBox, particles):
        gaussian_mse_map = compute_gaussian_mse_map_particle_set(image, prev_track_box, particles, sigma_mse)
        weights = np.asarray([gaussian_mse_map[y,x] for (x,y) in particles])
        weights = np.nan_to_num(weights,0)
        weights = weights / np.sum(weights) # normalizing to 1
        weights = np.nan_to_num(weights,0)
        return weights
    return get_weights

def get_diffuse_fn(sigma_noise=3):
    return lambda particles: diffuse(particles, sigma_noise)



In [93]:
# Mean Shift Lite
def chi_square_measure(h1, h2):
    if h1.shape != h2.shape:
        raise Exception("histograms are in different shapes", h1.shape, h2.shape)
    chi_square_value = 0
    for bin_count1, bin_count2 in zip(h1,h2):
        if (bin_count1 + bin_count2) != 0:
            chi_square_value += ((bin_count1 - bin_count2)**2) / (bin_count1 + bin_count2)

    return 0.5 * chi_square_value
        


def compute_RGB_diff_weights(image_rgb, prev_track_box: TrackBox, particles, num_bins_per_channel=8, normalization_factor=5):
    quantized_image = (image_rgb // (256 // num_bins_per_channel))
    quantized_image_prev = (prev_track_box.image // (256 // num_bins_per_channel))
    # Combine the quantized channels to get the final bin for each pixel
    bins_image = quantized_image[:, :, 0] * (num_bins_per_channel ** 2) + quantized_image[:, :, 1] * num_bins_per_channel + quantized_image[:, :, 2]
    bins_image_prev = quantized_image_prev[:, :, 0] * (num_bins_per_channel ** 2) + quantized_image_prev[:, :, 1] * num_bins_per_channel + quantized_image_prev[:, :, 2]
    # Now I have bin for each pixel and I want to count the bins for an histogram
    prev_bins_patch = get_patch(bins_image_prev, *prev_track_box.get_components())
    prev_box_histogram, _ = np.histogram(prev_bins_patch, bins=num_bins_per_channel**3, range=(0, num_bins_per_channel**3))

    chi_square_values = []
    for (x,y) in particles:
        bins_image_patch = get_patch(bins_image, x, y, prev_track_box.w, prev_track_box.h)
        histogram, _ = np.histogram(bins_image_patch, bins=num_bins_per_channel**3, range=(0, num_bins_per_channel**3))
        chi_square_value = chi_square_measure(histogram, prev_box_histogram)
        chi_square_values.append(chi_square_value)
    
    chi_square_values = np.array(chi_square_values)
    chi_square_values = np.power(math.e, -1 * chi_square_values / (2*normalization_factor**2))
    weights = chi_square_values / sum(chi_square_values)
    return weights

def get_measure_fn_mean_shift(num_bins=8, normalization_factor=5):
    def get_rgb_measure(image_rgb, prev_track_box: TrackBox, particles):
        return compute_RGB_diff_weights(image_rgb, prev_track_box, particles, num_bins, normalization_factor)
    return get_rgb_measure


In [161]:
particle_filter = ParticleFilterAlgo(frames_debate_bgr, 
                                     create_particles,
                                     sample_particles,
                                     get_diffuse_fn(10), 
                                    #  get_measure_fn(3),
                                     get_measure_fn_mean_shift(num_bins=3, normalization_factor=2),
                                     clip_into_frame, 
                                     num_particles=150,
                                     num_frames=50,)

In [162]:
states = particle_filter.apply_particle_filtering(debate_hand_box_path)
# states = particle_filter.apply_particle_filter_IIR(debate_hand_box_path, alpha=0.7)

100%|██████████| 50/50 [00:02<00:00, 17.57it/s]


In [163]:
import time
for state in states:
    frame_with_box = state.get_frame_with_particles_and_box()
    # frame_with_box = state.get_frame_with_box()
    
    cv2.imshow("frame",frame_with_box)
    # Press Q on keyboard to exit the display window
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break
    # time.sleep(0.5)
    
cv2.destroyAllWindows()
    